# 04 — LightGBM LambdaRank Backtest

Walk-forward evaluation of the LightGBM model vs rule-based scoring.

**Protocol (no leakage)**  
For each test date after the 30-day warm-up, the model is trained exclusively on all races *before* that date, then evaluated on that date's races.  
This mirrors live usage: every morning the model is retrained on the full history before generating recommendations.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from src.scraper import get_connection
from src.features.pipeline import compute_features
from src.model.backtest import backtest
from src.model.scorer import score_combined
from src.model.lgbm import backtest_lgbm_walkforward, train_lgbm, FEATURES

conn = get_connection()
df = compute_features(conn)
conn.close()

dates = sorted(df["date"].unique())
MIN_TRAIN = 30
test_dates = dates[MIN_TRAIN:]
test_df = df[df["date"].isin(test_dates)]

print(f"Full dataset : {len(dates)} days | {df['race_id'].nunique()} races | {len(df)} runners")
print(f"Train warm-up: {dates[0]} → {dates[MIN_TRAIN-1]} ({MIN_TRAIN} days)")
print(f"Test period  : {test_dates[0]} → {test_dates[-1]} ({len(test_dates)} days)")

## 1 · Walk-forward backtest

In [ ]:
# Rule-based — evaluated on the same test period (no retraining needed)
rb_win  = backtest(test_df, score_combined, "rule_based", bet_type="win")
rb_duo  = backtest(test_df, score_combined, "rule_based", bet_type="duo")

# LightGBM — walk-forward (retrained daily, no leakage)
lgbm_win = backtest_lgbm_walkforward(df, min_train_days=MIN_TRAIN, bet_type="win")
lgbm_duo = backtest_lgbm_walkforward(df, min_train_days=MIN_TRAIN, bet_type="duo")

rows = [
    {"Model": "Rule-based",  "Bet": "win", "Bets": len(rb_win.bets),  "ROI": rb_win.roi,  "Hit rate": rb_win.hit_rate,  "P&L": sum(b.pnl for b in rb_win.bets)},
    {"Model": "LightGBM",    "Bet": "win", "Bets": len(lgbm_win.bets), "ROI": lgbm_win.roi, "Hit rate": lgbm_win.hit_rate, "P&L": sum(b.pnl for b in lgbm_win.bets)},
    {"Model": "Rule-based",  "Bet": "duo", "Bets": len(rb_duo.bets),  "ROI": rb_duo.roi,  "Hit rate": rb_duo.hit_rate,  "P&L": sum(b.pnl for b in rb_duo.bets)},
    {"Model": "LightGBM",    "Bet": "duo", "Bets": len(lgbm_duo.bets), "ROI": lgbm_duo.roi, "Hit rate": lgbm_duo.hit_rate, "P&L": sum(b.pnl for b in lgbm_duo.bets)},
]
summary = pd.DataFrame(rows).set_index(["Model", "Bet"])
summary["ROI"] = summary["ROI"].map("{:.1%}".format)
summary["Hit rate"] = summary["Hit rate"].map("{:.1%}".format)
summary["P&L"] = summary["P&L"].map("{:+.1f}u".format)
summary

## 2 · With EV filter (model_prob > implied_prob)

In [ ]:
rb_ev   = backtest(test_df, score_combined, "rule_based_ev", bet_type="win", ev_filter=True)
lgbm_ev = backtest_lgbm_walkforward(df, min_train_days=MIN_TRAIN, bet_type="win", ev_filter=True)

rows_ev = [
    {"Model": "Rule-based + EV", "Bets": len(rb_ev.bets),   "ROI": rb_ev.roi,   "Hit rate": rb_ev.hit_rate,   "P&L": sum(b.pnl for b in rb_ev.bets)},
    {"Model": "LightGBM + EV",   "Bets": len(lgbm_ev.bets), "ROI": lgbm_ev.roi, "Hit rate": lgbm_ev.hit_rate, "P&L": sum(b.pnl for b in lgbm_ev.bets)},
]
ev_df = pd.DataFrame(rows_ev).set_index("Model")
ev_df["ROI"] = ev_df["ROI"].map("{:.1%}".format)
ev_df["Hit rate"] = ev_df["Hit rate"].map("{:.1%}".format)
ev_df["P&L"] = ev_df["P&L"].map("{:+.1f}u".format)
ev_df

## 3 · Cumulative P&L curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (rb, lgbm, title) in zip(axes, [
    (rb_win,  lgbm_win, "WIN bets — all races"),
    (rb_ev,   lgbm_ev,  "WIN bets — EV filter"),
]):
    rb.pnl_series().plot(ax=ax, label=f"Rule-based (ROI={rb.roi:.0%})", color="#e65100", linewidth=2)
    lgbm.pnl_series().plot(ax=ax, label=f"LightGBM (ROI={lgbm.roi:.0%})", color="#1565c0", linewidth=2)
    ax.axhline(0, color="#999", linewidth=0.8, linestyle="--")
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("Date")
    ax.set_ylabel("Cumulative P&L (units)")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")

plt.tight_layout()
plt.savefig("../data/reports/backtest_pnl_curve.png", dpi=150, bbox_inches="tight")
plt.show()

## 4 · Feature importance

In [ ]:
# Train on all available data (in-sample) for feature importance
full_model = train_lgbm(df)
importance = pd.Series(
    full_model.booster_.feature_importance(importance_type="gain"),
    index=FEATURES,
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
importance.plot.barh(ax=ax, color="#1565c0")
ax.set_title("Feature importance (gain) — full model", fontsize=13, fontweight="bold")
ax.set_xlabel("Gain")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()
print(importance.sort_values(ascending=False).to_string())

## 5 · Agreement between models

When both models pick the same horse, does it win more often?

In [ ]:
from src.model.lgbm import train_lgbm, score_lgbm
from src.model.scorer import score_combined

# Build per-race top-1 picks for both models over the test period
full_model = train_lgbm(df[df["date"] < test_dates[0]])  # trained on warm-up only

rows = []
for race_id, race_df in test_df.groupby("race_id"):
    race_df = race_df.copy()
    if len(race_df) < 4:
        continue

    rb_scores   = score_combined(race_df)
    lgbm_scores = score_lgbm(race_df, full_model)

    rb_pick   = rb_scores.idxmax()
    lgbm_pick = lgbm_scores.idxmax()
    agree = rb_pick == lgbm_pick

    winner = race_df[race_df["finish_position"] == 1]["runner_id"].values
    rb_won   = len(winner) > 0 and rb_pick   in winner
    lgbm_won = len(winner) > 0 and lgbm_pick in winner

    rows.append({"race_id": race_id, "agree": agree, "rb_won": rb_won, "lgbm_won": lgbm_won})

agree_df = pd.DataFrame(rows)
print(f"Races analysed : {len(agree_df)}")
print(f"Models agree   : {agree_df['agree'].sum()} ({agree_df['agree'].mean():.0%})")
print()
print("Hit rate when models AGREE :")
agreed = agree_df[agree_df["agree"]]
print(f"  Rule-based  : {agreed['rb_won'].mean():.1%} ({len(agreed)} races)")
print()
print("Hit rate when models DISAGREE :")
disagreed = agree_df[~agree_df["agree"]]
print(f"  Rule-based  : {disagreed['rb_won'].mean():.1%}")
print(f"  LightGBM    : {disagreed['lgbm_won'].mean():.1%}")